In [12]:
import pandas as pd
import sqlite3

df_path = "../data/raw/chess_games.csv"
db_path = "../data/processed/chess.db"

df = pd.read_csv(df_path)
conn = sqlite3.connect(db_path)

# Create the table from the CSV columns (rated, turns, winner, opening_code, ...)
df.to_sql("games", conn, if_exists="replace", index=False)
conn.commit()
conn.close()

print("Data loaded successfully!")
print("Columns:", list(df.columns))








Data loaded successfully!
Columns: ['game_id', 'rated', 'turns', 'victory_status', 'winner', 'time_increment', 'white_id', 'white_rating', 'black_id', 'black_rating', 'moves', 'opening_code', 'opening_moves', 'opening_fullname', 'opening_shortname', 'opening_response', 'opening_variation']


## Stage 1 — SELECT · Stage 2 — GROUP BY : Q1-Q6

In [ ]:
# 1.How many total games? How many are rated?
import sqlite3

conn = sqlite3.connect("../data/processed/chess.db")
cursor = conn.cursor()
cursor.execute("""
SELECT
    COUNT(*) AS total_games,
    SUM(CASE WHEN rated = 1 THEN 1 ELSE 0 END) AS rated_games
FROM games;
""")
print("1-",cursor.fetchall())
conn.commit()
conn.close()








In [17]:

import sqlite3
#2.List all distinct victory_status values and their counts.
conn = sqlite3.connect("../data/processed/chess.db")
cursor = conn.cursor()
cursor.execute("""
SELECT
    victory_status,
    COUNT(*) AS count
FROM games
GROUP BY victory_status
ORDER BY count DESC;
""")
print("2-",cursor.fetchall())
conn.commit()
conn.close()


2- [('Resign', 11147), ('Mate', 6325), ('Out of Time', 1680), ('Draw', 906)]


In [18]:
#3.The 10 games with the most turns. Show game_id, winner, turns.
conn = sqlite3.connect("../data/processed/chess.db")
cursor = conn.cursor()
cursor.execute("""
SELECT
    game_id,
    winner,
    turns
FROM games
ORDER BY turns DESC
LIMIT 10;
""")
print("3-",cursor.fetchall())
conn.commit()
conn.close()


3- [(11555, 'White', 349), (13860, 'White', 349), (16387, 'Draw', 259), (4237, 'Draw', 255), (16646, 'Draw', 226), (15479, 'Draw', 222), (16944, 'Black', 222), (6777, 'Draw', 221), (13231, 'Black', 218), (13556, 'Draw', 216)]


In [20]:
#4.What is the win rate (%) for White, Black, and Draw across all games?
import sqlite3
conn=sqlite3.connect("../data/processed/chess.db")
cursor=conn.cursor()
cursor.execute("""
SELECT 
    winner,
    COUNT(*) * 100.0 /(SELECT COUNT(*) FROM games) AS win_rate_percentage 
FROM games
GROUP BY winner;
""")
print("4-",cursor.fetchall())
conn.commit()
conn.close()


4- [('Black', 45.403330342008175), ('Draw', 4.736264831987237), ('White', 49.86040482600459)]


In [21]:
#5. For each victory_status, what is the average and max number of turns? Sort highest
# avg first.
import sqlite3
conn=sqlite3.connect("../data/processed/chess.db")
cursor=conn.cursor()
cursor.execute("""
SELECT
victory_status,
AVG(turns) AS avg_turns,
MAX(turns) AS max_turns
FROM games
GROUP BY victory_status
ORDER BY avg_turns DESC;
""")
print("5-",cursor.fetchall())
conn.commit()
conn.close()

5- [('Draw', 83.78145695364239, 259), ('Out of Time', 72.74285714285715, 349), ('Mate', 65.41501976284584, 222), ('Resign', 53.91253251996053, 218)]


In [22]:
# #6.Which 5 opening_codes appear most frequently? Use HAVING to show only those
# with more than 500 games.
import sqlite3
conn=sqlite3.connect("../data/processed/chess.db")
cursor=conn.cursor()
cursor.execute("""
SELECT
    opening_code,
    COUNT(*) AS count
FROM games
GROUP BY opening_code
HAVING count > 500
ORDER BY count DESC
LIMIT 5;
""")
print("6-",cursor.fetchall())
conn.commit()
conn.close()


6- [('A00', 1007), ('C00', 844), ('D00', 739), ('B01', 716), ('C41', 691)]


#### Question Answers 
 1- [(20058, 16155)]

 2- [('Resign', 11147), ('Mate', 6325), ('Out of Time', 1680), ('Draw', 906)]

 3- [(11555, 'White', 349), (13860, 'White', 349), (16387, 'Draw', 259), (4237, 'Draw', 255), (16646, 'Draw', 226), (15479, 'Draw', 222), (16944, 'Black', 222), (6777, 'Draw', 221), (13231, 'Black', 218), (13556, 'Draw', 216)]

4- [('Black', 45.403330342008175), ('Draw', 4.736264831987237), ('White', 49.86040482600459)]

5- [('Draw', 83.78145695364239, 259), ('Out of Time', 72.74285714285715, 349), ('Mate', 65.41501976284584, 222), ('Resign', 53.91253251996053, 218)]

6- [('A00', 1007), ('C00', 844), ('D00', 739), ('B01', 716), ('C41', 691)]


### Reset DB Pipline 


In [ ]:
import sqlite3

def reset_database(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # 1. Turn off Foreign Key checks so we can drop tables in any order
    cursor.execute('PRAGMA foreign_keys = OFF')
    
    # 2. Drop existing tables
    # List them in order of importance
    cursor.execute('DROP TABLE IF EXISTS games')
    cursor.execute('DROP TABLE IF EXISTS players')

    conn.commit()
    conn.close()
    print("Database cleaned and schema recreated.")

# Run this at the start of your development workflow
reset_database('../data/processed/chess.db')